In [ ]:
%env AWS_PROFILE=kmnlp

In [ ]:
from rich import print as rprint
from rich.pretty import pretty_repr

## Dynamical.org

https://dynamical.org/catalog/


### ECMWF IFS ENS Forecast, 15 day, 0.25 degree


In [ ]:
import xarray as xr

In [ ]:
uri = "https://data.dynamical.org/ecmwf/ifs-ens/forecast-15-day-0-25-degree/latest.zarr"

In [ ]:
ds_forecast = xr.open_zarr(uri, chunks="auto")  # pyright: ignore[reportUnknownMemberType]
ds_forecast

### NOAA GFS analysis, hourly


In [ ]:
uri = "https://data.dynamical.org/noaa/gfs/analysis-hourly/latest.zarr"

In [ ]:
ds_zarr = xr.open_zarr(uri, chunks="auto")  # pyright: ignore[reportUnknownMemberType]
ds_zarr

In [ ]:
import cartopy  # Map projections libary
import cartopy.crs as ccrs  # pyright: ignore[reportMissingTypeStubs] # Projections list
from matplotlib import pyplot as plt


In [ ]:
da = ds_zarr.isel(time=-100)

In [ ]:
plt.close("all")
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})  # pyright: ignore[reportUnknownMemberType]
ax.quiver(  # pyright: ignore[reportUnknownMemberType]
    ds_zarr.longitude[::32],
    ds_zarr.latitude[::32],
    da["wind_u_10m"][::32, ::32],
    da["wind_u_10m"][::32, ::32],
    transform=ccrs.PlateCarree(),
)
ax.coastlines(color="grey")  # pyright: ignore[reportAttributeAccessIssue, reportUnknownMemberType]
plt.show()  # pyright: ignore[reportUnknownMemberType]

In [ ]:
plt.close("all")
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.EqualEarth()})  # pyright: ignore[reportUnknownMemberType]
ax.streamplot(
    ds_zarr.longitude,
    ds_zarr.latitude,
    da["wind_u_10m"],
    da["wind_u_10m"],
    density=1,
    transform=ccrs.PlateCarree(),
)
ax.coastlines(color="grey")  # pyright: ignore[reportAttributeAccessIssue, reportUnknownMemberType]
plt.show()  # pyright: ignore[reportUnknownMemberType]

## Earthmover

https://docs.earthmover.io/sample-data


In [ ]:
!uv pip install arraylake

### ERA5


Once you have created an Earthmover account:


In [ ]:
!arraylake auth login

In [ ]:
import xarray as xr
from arraylake import Client


In [ ]:
client = Client()
repo = client.get_repo("earthmover-public/era5-surface-aws")
session = repo.readonly_session(branch="main")

In [ ]:
era5 = xr.open_zarr(session.store, zarr_format=3, group="spatial")  # pyright: ignore[reportUnknownMemberType]
era5

In [ ]:
era5_t = xr.open_zarr(session.store, zarr_format=3, group="temporal")  # pyright: ignore[reportUnknownMemberType]
era5_t

In [ ]:
era5_t["u10"].isel(time=-1).plot()  # pyright: ignore[reportCallIssue]

## Google


In [ ]:
!uv pip install intake-esm gcsfs

### CMIP6


In [ ]:
import intake
import pandas as pd
import xarray as xr

# URL for the Google Cloud CMIP6 ESM Collection Spec (JSON format is also available)
# The CSV is often easier to load directly into a pandas DataFrame for simple searching
catalog_url = "https://cmip6.storage.googleapis.com/pangeo-cmip6.csv"
df = pd.read_csv(catalog_url)  # pyright: ignore[reportUnknownMemberType]

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df["activity_id"].unique()

In [ ]:
df_subset = df.query(  # pyright: ignore[reportUnknownMemberType]
    "activity_id=='CMIP' & table_id=='Amon' & variable_id=='tas' & source_id=='MPI-ESM1-2-LR'"
)
print(f"Found {len(df_subset)} datasets matching the criteria.")

In [ ]:
df.variable_id.unique()

In [ ]:
df.table_id.unique()

In [ ]:
import gcsfs

# Authenticate for anonymous public access to Google Cloud Storage
gcs = gcsfs.GCSFileSystem(token="anon")  # noqa: S106

# Get the path to a specific zarr store (e.g., the first one in the subset)
zstore_url = df_subset.zstore.to_numpy()[0]

# Use the gcsfs mapper to interface with the Zarr store
mapper = gcs.get_mapper(zstore_url)  # pyright: ignore[reportUnknownMemberType]

# Open the Zarr store using xarray
# The 'consolidated=True' argument helps read the metadata efficiently
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
ds = xr.open_zarr(mapper, consolidated=True, decode_times=time_coder)  # pyright: ignore[reportArgumentType, reportUnknownMemberType]

# Now you can work with the xarray dataset (ds)
ds


## NODD

s3://noaa-nodd-kerchunk-pds/


In [ ]:
import xarray as xr

In [ ]:
# uri = "s3://noaa-nodd-kerchunk-pds/nos/cbofs/cbofs.fields.best.nc.zarr"
uri = "s3://noaa-nodd-kerchunk-pds/nwm_ll/LatLon_hawaii_v3.0.nc.zarr"
ds = xr.open_dataset(uri, engine="kerchunk", chunks="auto")  # pyright: ignore[reportUnknownMemberType]
ds